# EMA + RSI

## Contents

- [Configuration](#configuration)
  - [Setup](#setup)
  - [Automatic](#automatic)
  - [Manual](#manual)
  - [Final configuration](#final-configuration)
- [EMA + RSI](#ema-rsi-section)
  - [Backtesting](#backtesting)
  - [Grid search](#grid-search)
  - [Walk-forward analysis](#walk-forward-analysis)
  - [Monte Carlo simulations](#monte-carlo-simulations)
  - [Live signals](#live-signals)
- [Inverse EMA + RSI](#inverse-ema--rsi)
  - [Backtesting](#inv-backtesting)
  - [Grid search](#inv-grid-search)
  - [Walk-forward analysis](#inv-walk-forward)
  - [Monte Carlo simulations](#inv-monte-carlo)
  - [Live signals](#inv-live-signals)
- [Adaptive EMA + RSI](#adaptive-ema--rsi)
  - [Backtesting](#adp-backtesting)
  - [Grid search](#adp-grid-search)
  - [Walk-forward analysis](#adp-walk-forward)
  - [Monte Carlo simulations](#adp-monte-carlo)
  - [Live signals](#adp-live-signals)

EMA Crossover + RSI Filter \
A momentum strategy that trades in the direction of the cross: fast EMA(9) over slow EMA(21) \
It uses ATR(14) for a dynamic volatility-adaptive trailing stop. \
Filters false signals with RSI(14) to cut whipsaws in ranging markets. \
Works across timeframes; well suited to scalping and intraday.

__How EMA+RSI Algorithm Determines Entry/Exit:__
- Fast EMA (9) / Slow EMA (21) – standard for crypto.
- Long Entry: Fast EMA crosses above Slow EMA AND RSI(14) < 70 (not overbought).
- Short Entry: Fast EMA crosses below Slow EMA AND RSI(14) > 30 (not oversold).
- Exit: Reverse crossover (signal flip) OR price hits the ATR-based trailing stop.
- The RSI filter reduces whipsaws in ranging markets.

__The RSI filter__ on ema / ema_inv is an optional, configurable filter:
- filter off entirely: EmaParams(rsi_filter=False)
- custom bounds: EmaParams(rsi_bullish=65.0, rsi_bearish=35.0)

## Configuration

### Setup

In [1]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
import dataclasses

from engine.backtester import Backtester
from engine.data_configurator import ACTIVE, load_data, save_result, LIVE_DIR
from engine.strategy_configurator import params_for, EXIT_PRESETS
from engine.trade_configurator import ACTIVE_TRADE
from engine.visualization import build_chart
from engine.evaluation import walk_forward, monte_carlo, grid_search
from engine.live import LiveEngine

import pandas as pd
import plotly.express as px

### Automatic

In [3]:
# Automatic config: project-wide defaults defined by the three configurators.
# Override any of them in the manual chapter below; leave its dicts empty to stay automatic.
DATA_CONFIG     = ACTIVE              # engine/data_configurator.py     (DataSpec)
STRATEGY_CONFIG = params_for("ema")   # engine/strategy_configurator.py (EmaParams — this notebook's family)
TRADING_CONFIG  = ACTIVE_TRADE        # engine/trade_configurator.py    (TradingConfig)
EXIT_POLICY     = None                # None -> each strategy's assigned default (exit_policy_for)

### Manual


*_CONFIG = Automatic defaults, with any Manual overrides layered on top:
- Leave *_OVERRIDES empty → *_CONFIG is pure Automatic.
- Fill it → Automatic baseline + your Manual overrides.

In [4]:
# manual override — DATA. Fill the dict to override; empty = automatic (ACTIVE).
DATA_OVERRIDES = {}          # e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

In [5]:
# manual override — STRATEGY signal knobs. Empty = automatic EmaParams().
STRATEGY_OVERRIDES = {}      # e.g. {"ema_fast": 12, "ema_slow": 26, "rsi_filter": False}
STRATEGY_CONFIG = dataclasses.replace(STRATEGY_CONFIG, **STRATEGY_OVERRIDES)

In [6]:
# manual override — EXIT policy. None = each strategy's assigned default.
# Preset: EXIT_POLICY = EXIT_PRESETS["fixed_2pct_rr3"]()
# Custom: from engine.exits import CompositeExit, AtrStop, RrTarget
#         EXIT_POLICY = CompositeExit(AtrStop(1.5), RrTarget(2.0))
EXIT_POLICY = None

In [7]:
# manual override — TRADE config (costs / sizing / leverage / direction). Empty = ACTIVE_TRADE.
# For direction: from engine.trade_configurator import TradeDirection
TRADE_OVERRIDES = {}         # e.g. {"leverage": 2.0, "direction": TradeDirection.LONG}
TRADING_CONFIG = dataclasses.replace(TRADING_CONFIG, **TRADE_OVERRIDES)

### Final configuration

In [8]:
# Prepare the final inputs the rest of the notebook uses.
# Runs after overrides.
df = load_data(DATA_CONFIG)
SYMBOL, INTERVAL = DATA_CONFIG.symbol, DATA_CONFIG.interval

In [9]:
# Report exactly what data + which configs are in force downstream (manual or automatic).
_window = (f"{DATA_CONFIG.start} → {DATA_CONFIG.end or 'now'}"
           if DATA_CONFIG.is_range else f"last {DATA_CONFIG.num_candles}")
tc = TRADING_CONFIG
_exits = (", ".join(f"{k!r}: {v!r}" for k, v in STRATEGY_CONFIG.EXITS.items())
          if EXIT_POLICY is None else f"override → {EXIT_POLICY}")
print(f"Loaded {len(df):,} candles | {SYMBOL} {INTERVAL}m {DATA_CONFIG.category} | "
      f"{_window} | {df.index[0]:%Y-%m-%d %H:%M} → {df.index[-1]:%Y-%m-%d %H:%M} UTC")
print(f"Trade: initial_equity={tc.initial_equity}, position_size_bps={tc.position_size_bps}, "
      f"leverage={tc.leverage}, sizing_mode={tc.sizing_mode.value!r}, "
      f"risk_per_trade_bps={tc.risk_per_trade_bps}, direction={tc.direction.value!r}")
print("Strategy Parameters: "
      + ", ".join(f"{k}={v}" for k, v in dataclasses.asdict(STRATEGY_CONFIG).items()))
print(f"Strategy exits: {_exits}")

Loaded 800 candles | BTCUSDT 15m linear | last 800 | 2026-06-04 15:00 → 2026-06-12 22:45 UTC
Trade: initial_equity=10000.0, position_size_bps=10000.0, leverage=1.0, sizing_mode='fixed', risk_per_trade_bps=100.0, direction='both'
Strategy Parameters: atr_period=14, ema_fast=9, ema_slow=21, rsi_period=14, rsi_filter=True, rsi_bullish=70.0, rsi_bearish=30.0
Strategy exits: 'ema': 'chandelier_2atr', 'ema_inv': 'chandelier_2atr', 'ema_adaptive': 'chandelier_2atr'


<a id="ema-rsi-section"></a>
## EMA + RSI

### Backtesting

In [10]:
# Import EMA + RSI strategy
from engine.strategies import EMACrossoverStrategy
STRATEGY = EMACrossoverStrategy

In [11]:
# Backtest EMA + RSI strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Per-trade dollar P&L
trades_pnl = pd.DataFrame([{
    "dir": t.direction.value,
    "pnl_bps": round(t.pnl_bps, 1),
    "pnl_$": round(t.pnl_currency, 2),
    "balance_after": round(t.equity_after, 2),
    "avg_duration_min": round(t.duration.total_seconds() / 60, 1) if t.duration else None,
    "exit_reason": t.exit_reason.value if t.exit_reason else None,
} for t in result.trades])
display(trades_pnl.head())

# Save metrics (JSON) + per-trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG, name="ema_rsi")

════════════════════════════════════════════════════════════
  Backtest Summary: ema
  BTCUSDT | 15 | 800 bars
════════════════════════════════════════════════════════════
  Total trades      : 32
  Avg duration min  : 160.8
  Suppressed entries : 0  (blocked by direction/daily-loss gate)
  Win / Loss / BE    : 7 / 25 / 0
  Win rate           : 21.9%
  Total P&L (bps)    : -1233.8
  Avg P&L (bps)      : -38.6
  Max win (bps)      : +189.1
  Max loss (bps)     : -122.5
  Profit factor      : 0.27
  Max drawdown (bps) : 1160.7
  Sharpe (approx)    : -0.55
  ────────────────────────────────────────
  Initial balance    : $10,000.00
  Final balance      : $8,830.52
  Net profit         : $-1,169.48
  Net return         : -11.69%
  Max drawdown       : 11.85%
  ────────────────────────────────────────
  Exits by reason:
    trailing_stop    : 24
    signal_flip      : 7
    force_close      : 1
════════════════════════════════════════════════════════════


,dir,pnl_bps,pnl_$,balance_after,avg_duration_min,exit_reason
0,long,-94.5,-94.46,9905.54,75.0,signal_flip
1,long,-109.3,-108.28,9797.26,15.0,signal_flip
2,long,-103.6,-101.53,9695.73,90.0,signal_flip
3,long,-122.5,-118.82,9576.91,150.0,trailing_stop
4,short,10.4,9.98,9586.89,195.0,trailing_stop


PosixPath('/Users/gm/Projects/tradekit/data/results/linear_BTCUSDT_15_last800/ema_rsi.json')

In [12]:
# EMA + RSI strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

### Grid search

In [ ]:
# Full grid search — Cartesian product across any of the four dimensions.
# Each grid is optional: uncomment the ones you want to sweep, leave the rest commented to hold them fixed.
# Row count = |strategy| × |trade| × |exit| × |data|, so keep grids tight.
# Rank by total_return_pct: pnl_bps is sizing-invariant, so trade-knob sweeps only move equity.

gs = grid_search(
    STRATEGY,
    strategy_grid={"ema_fast": [19, 21, 56], "ema_slow": [22, 30, 76]},
    trade_grid={"leverage": [1.0, 2.0]},
    exit_grid=[None, "fixed_2pct_rr3", "chandelier_2atr"],
    data_grid={"interval": ["15", "60"]},   # reloads per spec
    base_config=STRATEGY_CONFIG, base_trading=TRADING_CONFIG, base_data=DATA_CONFIG,
)
gs.sort_values("total_return_pct", ascending=False).head(12)

,interval,ema_fast,ema_slow,leverage,exit,trades,win_rate,profit_factor,max_drawdown_bps,sharpe_approx,total_return_pct,total_pnl_bps,net_profit_usd,final_equity
94,60,56,22,2.0,fixed_2pct_rr3,10,0.800000,2.572631,212.000000,0.372764,13.522108,666.795451,1352.210825,11352.210825
100,60,56,30,2.0,fixed_2pct_rr3,9,0.777778,2.473315,212.000000,0.384898,12.630300,624.685492,1263.029998,11263.029998
91,60,56,22,1.0,fixed_2pct_rr3,10,0.800000,2.572631,212.000000,0.372764,6.719377,666.795451,671.937661,10671.937661
97,60,56,30,1.0,fixed_2pct_rr3,9,0.777778,2.473315,212.000000,0.384898,6.285702,624.685492,628.570242,10628.570242
52,15,56,76,2.0,fixed_2pct_rr3,4,0.500000,3.009584,131.104662,0.433484,5.232187,263.465865,523.218652,10523.218652
45,15,56,30,2.0,default,7,0.571429,1.737871,96.607624,0.185212,3.483482,183.397118,348.348175,10348.348175
47,15,56,30,2.0,chandelier_2atr,7,0.571429,1.737871,96.607624,0.185212,3.483482,183.397118,348.348175,10348.348175
49,15,56,76,1.0,fixed_2pct_rr3,4,0.500000,3.009584,131.104662,0.433484,2.625715,263.465865,262.571502,10262.571502
41,15,56,22,2.0,chandelier_2atr,8,0.375000,1.351440,96.607624,0.106811,2.474355,140.878510,247.435521,10247.435521
39,15,56,22,2.0,default,8,0.375000,1.351440,96.607624,0.106811,2.474355,140.878510,247.435521,10247.435521


In [ ]:
# HEATMAP_METRIC is a configurable knob:
# flip between Sharpe / P&L / profit_factor / any grid_search column without editing the plot.
# Best value per ema_fast × ema_slow cell, across any other swept dimension.
# Renders only when the strategy grid is being swept.

HEATMAP_METRIC = "total_pnl_bps"   # any grid_search column
if {"ema_fast", "ema_slow"}.issubset(gs.columns):
    # Diverging colour split at the metric's breakeven: P&L/Sharpe at 0, profit_factor at 1, win_rate at 0.5.
    midpoint = {"profit_factor": 1.0, "win_rate": 0.5}.get(HEATMAP_METRIC, 0.0)
    px.imshow(
        gs.pivot_table(index="ema_fast", columns="ema_slow", values=HEATMAP_METRIC, aggfunc="max"),
        color_continuous_scale="RdYlGn", color_continuous_midpoint=midpoint, aspect="auto",
        labels=dict(x="ema_slow", y="ema_fast", color=HEATMAP_METRIC),
        title=f"In-sample {HEATMAP_METRIC} — {strategy.name} | {SYMBOL} {INTERVAL}m",
    ).show()
else:
    print("Heatmap needs ema_fast × ema_slow swept in the grid above — nothing to plot.")

### Walk-forward analysis

In [15]:
# Walk-forward re-sweeps the strategy grid every train window.
# TRAIN_BARS is an in-sample window swept for the best params.
# TEST_BARS is an out-of-sample window the winner is then tested on.
# OBJECTIVE  can be any sweep metric: total_pnl_bps | sharpe_approx | profit_factor | ...
# MIN_TRADES lets ignore in-sample combos with fewer trades (noise, not signal)

GRID = {"ema_fast": [5, 9, 13, 17], "ema_slow": [20, 30, 40, 50]}
MIN_TRADES = 2
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    base_config=STRATEGY_CONFIG,   # non-swept knobs come from STRATEGY_CONFIG (automatic + manual)
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

════════════════════════════════════════════════════════════
  Walk-Forward (rolling) — 5 folds, objective=total_pnl_bps
  train=300 bars / test=100 bars
════════════════════════════════════════════════════════════
  OOS trades         : 13
  OOS total P&L (bps): -156.6
  OOS win rate       : 30.8%
  OOS profit factor  : 0.64
  OOS max DD (bps)   : 217.8
  OOS Sharpe (approx): -0.16
  ────────────────────────────────────────
  Σ in-sample      P&L (bps): -292.8
  Σ out-of-sample  P&L (bps): -156.6
  WF efficiency (OOS/IS): n/a   (in-sample best was unprofitable — no edge to carry over)
  ────────────────────────────────────────
  OOS equity (compounded, full balance per trade, no leverage):
  Initial balance    : $10,000.00
  Final balance      : $9,841.12
  Net profit         : $-158.88
  Net return         : -1.59%
  Max drawdown       : 2.16%
════════════════════════════════════════════════════════════


In [ ]:
# Per fold: the parameters chosen in-sample (is) and their out-of-sample (oos) performance.
# Parameters that vary significantly across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

,fold,test_start,test_end,ema_fast,ema_slow,is_total_pnl_bps,oos_total_pnl_bps,oos_trades,oos_win_rate
0,0,2026-06-07 18:00:00+00:00,2026-06-08 18:45:00+00:00,17,50,-118.243311,0.000000,0,0.000000
1,1,2026-06-08 19:00:00+00:00,2026-06-09 19:45:00+00:00,13,50,-242.953854,51.144164,4,0.500000
2,2,2026-06-09 20:00:00+00:00,2026-06-10 20:45:00+00:00,17,50,-62.780699,-20.628350,1,0.000000
3,3,2026-06-10 21:00:00+00:00,2026-06-11 21:45:00+00:00,13,50,59.726309,-70.586258,3,0.333333
4,4,2026-06-11 22:00:00+00:00,2026-06-12 22:45:00+00:00,5,50,71.491093,-116.520066,5,0.200000


In [ ]:
# Best parameters per fold (top) + cross-fold stability summary (bottom).
# Stable across folds = trustworthy; jumpy = tend to overfit, unlikely to work next window.
# nunique==1 => the optimiser locked the same value every fold; wide min..max / large std => jumpy.
display(wf.param_stability())
wf.param_stability_summary()

,ema_fast,ema_slow
fold,,
0,17,50
1,13,50
2,17,50
3,13,50
4,5,50


,nunique,mode,min,max,std
ema_fast,3,13.0,5,17,4.38178
ema_slow,1,50.0,50,50,0.00000


In [18]:
# Out-of-sample equity curve: the stitched test windows compounded.
# The equity path you'd have lived through re-tuning periodically on only past data.

eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
(px.line(eq, labels={"value": "equity", "index": ""}, color_discrete_sequence=["steelblue"],
         title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m")
 .update_layout(showlegend=False)
 .update_traces(hovertemplate="%{x}<br>%{y:,.2f}<extra></extra>",
                hoverlabel=dict(bgcolor="steelblue", font_color="white"))
 .show())

In [ ]:
# Walk-forward OOS trades (entries/exits), drawn with the EMAs each fold actually traded.
# Entries sit on real crosses: each fold's winning EMAs are recomputed and shown only over that fold's test window.
# The lines step at fold boundaries (the visible jump) is the re-optimisation.
# See wf.folds_frame() for the per-window parameters.

prepared_wf = df.copy()
prepared_wf["ema_fast"] = float("nan")
prepared_wf["ema_slow"] = float("nan")
for f in wf.folds:
    prep = STRATEGY(dataclasses.replace(STRATEGY_CONFIG, **f.best_params)).prepare(df)
    seg = (df.index >= f.test_start) & (df.index <= f.test_end)
    prepared_wf.loc[seg, ["ema_fast", "ema_slow"]] = prep.loc[seg, ["ema_fast", "ema_slow"]]

build_chart(prepared_wf, trades=wf.oos_trades,
            title=f"Walk-forward OOS trades — {strategy.name} | per-fold EMAs").show()

### Monte Carlo simulations

In [20]:
# The spread shows the range of results that could occur due to luck, not just the outcome that actually happened.
# It answers: how much did trade ordering affect the result, and how severe could the drawdown realistically be?
# P(profitable) near 50% means the OOS edge is equivalent to noise (close to a random chance).

mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

════════════════════════════════════════════════════════════
  Monte Carlo — 10000 sims, block=5, 13 trades
════════════════════════════════════════════════════════════
  Terminal return %  : median -1.7   [p5 -4.6 … p95 +1.6]
  Max drawdown %     : median 2.8    [p5 1.4 … p95 4.8]
  P(profitable)      : 22.0%
════════════════════════════════════════════════════════════


In [ ]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.

px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"}, color_discrete_sequence=["steelblue"],
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()

# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"}, color_discrete_sequence=["steelblue"],
             title="OOS max-drawdown distribution").show()

### Live signals

In [ ]:
# Live mode runs the same strategy / config / costs as the backtest above.
# It generates signals (tells you when to enter / exit). It does not place orders.
# run() prints a clickable link to the chart, which auto-refreshes every poll_seconds.
# engine.run() blocks the execution of the rest of the notebook until stopped.

live_strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
engine = LiveEngine(
    strategy=live_strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=10,
    trading_config=TRADING_CONFIG,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{live_strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{live_strategy.name}.db"),
)
engine.run()    # to stop: interrupt the kernel/ use the Stop button

## Inverse EMA + RSI

Inverse EMA Crossover + RSI Filter \
Mean-reversion counterpart to EMA+RSI: fades the cross instead of riding it. \
Uses ATR(14) for dynamic trailing stops (adapts to volatility) and RSI(14) filter. \
Bets that EMA crosses mark momentum exhaustion, not continuation.

__How Inverse EMA+RSI Algorithm Determines Entry/Exit:__
- Fast EMA (9) / Slow EMA (21) — standard for crypto.
- Short Entry: Fast EMA crosses above Slow EMA AND RSI(14) > 30 (not oversold) — fading the bullish cross.
- Long Entry:  Fast EMA crosses below Slow EMA AND RSI(14) < 70 (not overbought) — fading the bearish cross.
- Exit: Opposite crossover (signal flip) OR price hits ATR-based trailing stop.
- Works best in range-bound / mean-reverting regimes; likely underperforms in strong trends.


<a id="inv-backtesting"></a>
### Backtesting

In [ ]:
# Import Inverse EMA + RSI strategy
from engine.strategies import InverseEMACrossoverStrategy
STRATEGY = InverseEMACrossoverStrategy

In [ ]:
# Backtest Inverse EMA + RSI strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Per-trade dollar P&L
trades_pnl = pd.DataFrame([{
    "dir": t.direction.value,
    "pnl_bps": round(t.pnl_bps, 1),
    "pnl_$": round(t.pnl_currency, 2),
    "balance_after": round(t.equity_after, 2),
    "avg_duration_min": round(t.duration.total_seconds() / 60, 1) if t.duration else None,
    "exit_reason": t.exit_reason.value if t.exit_reason else None,
} for t in result.trades])
display(trades_pnl.head())

# Save metrics (JSON) + per-trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG, name="ema_rsi_inv")

In [ ]:
# Inverse EMA + RSI strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

<a id="inv-grid-search"></a>
### Grid search

In [ ]:
# Full grid search — Cartesian product across any of the four dimensions.
# Each grid is optional: uncomment the ones you want to sweep, leave the rest commented to hold them fixed.
# Row count = |strategy| × |trade| × |exit| × |data|, so keep grids tight.
# Rank by total_return_pct: pnl_bps is sizing-invariant, so trade-knob sweeps only move equity.

gs = grid_search(
    STRATEGY,
    strategy_grid={"ema_fast": [19, 21, 56], "ema_slow": [22, 30, 76]},
    trade_grid={"leverage": [1.0, 2.0]},
    exit_grid=[None, "fixed_2pct_rr3", "chandelier_2atr"],
    data_grid={"interval": ["15", "60"]},   # reloads per spec
    base_config=STRATEGY_CONFIG, base_trading=TRADING_CONFIG, base_data=DATA_CONFIG,
)
gs.sort_values("total_return_pct", ascending=False).head(12)

In [ ]:
# HEATMAP_METRIC is a configurable knob:
# flip between Sharpe / P&L / profit_factor / any grid_search column without editing the plot.
# Best value per ema_fast × ema_slow cell, across any other swept dimension.
# Renders only when the strategy grid is being swept.

HEATMAP_METRIC = "total_pnl_bps"   # any grid_search column
if {"ema_fast", "ema_slow"}.issubset(gs.columns):
    # Diverging colour split at the metric's breakeven: P&L/Sharpe at 0, profit_factor at 1, win_rate at 0.5.
    midpoint = {"profit_factor": 1.0, "win_rate": 0.5}.get(HEATMAP_METRIC, 0.0)
    px.imshow(
        gs.pivot_table(index="ema_fast", columns="ema_slow", values=HEATMAP_METRIC, aggfunc="max"),
        color_continuous_scale="RdYlGn", color_continuous_midpoint=midpoint, aspect="auto",
        labels=dict(x="ema_slow", y="ema_fast", color=HEATMAP_METRIC),
        title=f"In-sample {HEATMAP_METRIC} — {strategy.name} | {SYMBOL} {INTERVAL}m",
    ).show()
else:
    print("Heatmap needs ema_fast × ema_slow swept in the grid above — nothing to plot.")

<a id="inv-walk-forward"></a>
### Walk-forward analysis

In [ ]:
# Walk-forward re-sweeps the strategy grid every train window.
GRID = {"ema_fast": [5, 9, 13, 17], "ema_slow": [20, 30, 40, 50]}
MIN_TRADES = 2
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    base_config=STRATEGY_CONFIG,   # non-swept knobs come from STRATEGY_CONFIG (automatic + manual)
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

In [ ]:
# Per fold: the parameters chosen in-sample (is) and their out-of-sample (oos) performance.
# Parameters that vary significantly across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

In [ ]:
# Best parameters per fold (top) + cross-fold stability summary (bottom).
# Stable across folds = trustworthy; jumpy = tend to overfit, unlikely to work next window.
# nunique==1 => the optimiser locked the same value every fold; wide min..max / large std => jumpy.
display(wf.param_stability())
wf.param_stability_summary()

In [ ]:
# Out-of-sample equity curve: the stitched test windows compounded.
# The equity path you'd have lived through re-tuning periodically on only past data.

eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
(px.line(eq, labels={"value": "equity", "index": ""}, color_discrete_sequence=["steelblue"],
         title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m")
 .update_layout(showlegend=False)
 .update_traces(hovertemplate="%{x}<br>%{y:,.2f}<extra></extra>",
                hoverlabel=dict(bgcolor="steelblue", font_color="white"))
 .show())

In [ ]:
# Walk-forward OOS trades (entries/exits), drawn with the EMAs each fold actually traded.
# Entries sit on real crosses: each fold's winning EMAs are recomputed and shown only over that fold's test window.
# The lines step at fold boundaries (the visible jump) is the re-optimisation.
# See wf.folds_frame() for the per-window parameters.

prepared_wf = df.copy()
prepared_wf["ema_fast"] = float("nan")
prepared_wf["ema_slow"] = float("nan")
for f in wf.folds:
    prep = STRATEGY(dataclasses.replace(STRATEGY_CONFIG, **f.best_params)).prepare(df)
    seg = (df.index >= f.test_start) & (df.index <= f.test_end)
    prepared_wf.loc[seg, ["ema_fast", "ema_slow"]] = prep.loc[seg, ["ema_fast", "ema_slow"]]

build_chart(prepared_wf, trades=wf.oos_trades,
            title=f"Walk-forward OOS trades — {strategy.name} | per-fold EMAs").show()

<a id="inv-monte-carlo"></a>
### Monte Carlo simulations

In [ ]:
# The spread shows the range of results that could occur due to luck, not just the outcome that actually happened.
# It answers: how much did trade ordering affect the result, and how severe could the drawdown realistically be?
# P(profitable) near 50% means the OOS edge is equivalent to noise (close to a random chance).

mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

In [ ]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.

px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"}, color_discrete_sequence=["steelblue"],
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()

# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"}, color_discrete_sequence=["steelblue"],
             title="OOS max-drawdown distribution").show()

<a id="inv-live-signals"></a>
### Live signals

In [ ]:
# Live mode runs the same strategy / config / costs as the backtest above.
# It generates signals (tells you when to enter / exit). It does not place orders.
# run() prints a clickable link to the chart, which auto-refreshes every poll_seconds.
# engine.run() blocks the execution of the rest of the notebook until stopped.

live_strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
engine = LiveEngine(
    strategy=live_strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=10,
    trading_config=TRADING_CONFIG,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{live_strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{live_strategy.name}.db"),
)
engine.run()    # to stop: interrupt the kernel/ use the Stop button

<a id="adaptive-ema--rsi"></a>
## Adaptive EMA + RSI

RSI-driven regime switch — the EMA analogue of Adaptive SuperTrend's ADX switch. \
The EMA fast/slow cross is the raw signal; RSI decides whether to follow that cross or fade it:
- bullish cross + RSI not overbought (rsi < rsi_bullish) → follow → long
- bullish cross + RSI overbought (rsi ≥ rsi_bullish) → fade → short
- bearish cross + RSI not oversold (rsi > rsi_bearish) → follow → short
- bearish cross + RSI oversold (rsi ≤ rsi_bearish) → fade → long

So the same RSI bounds the plain ema strategy uses to skip an entry are used here to flip it: an overbought bullish cross becomes a mean-reversion short. Exits respect the regime captured at entry (follow vs fade), so a mid-trade RSI swing can't change a position's exit. \
Tune the switch from the manual chapter via the RSI bounds — e.g. STRATEGY_OVERRIDES = {"rsi_bullish": 65, "rsi_bearish": 35}. rsi_filter=False turns the switch off (plain follow-the-cross, same as base ema).

<a id="adp-backtesting"></a>
### Backtesting

In [ ]:
# Import Adaptive EMA + RSI strategy
from engine.strategies import AdaptiveEMACrossoverStrategy
STRATEGY = AdaptiveEMACrossoverStrategy

In [ ]:
# Backtest Adaptive EMA + RSI strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Per-trade dollar P&L
trades_pnl = pd.DataFrame([{
    "dir": t.direction.value,
    "pnl_bps": round(t.pnl_bps, 1),
    "pnl_$": round(t.pnl_currency, 2),
    "balance_after": round(t.equity_after, 2),
    "avg_duration_min": round(t.duration.total_seconds() / 60, 1) if t.duration else None,
    "exit_reason": t.exit_reason.value if t.exit_reason else None,
} for t in result.trades])
display(trades_pnl.head())

# Save metrics (JSON) + per-trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG, name="ema_rsi_adaptive")

In [ ]:
# Adaptive EMA + RSI strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

<a id="adp-grid-search"></a>
### Grid search

In [ ]:
# Full grid search — Cartesian product across any of the four dimensions.
# Each grid is optional: uncomment the ones you want to sweep, leave the rest commented to hold them fixed.
# Row count = |strategy| × |trade| × |exit| × |data|, so keep grids tight.
# Rank by total_return_pct: pnl_bps is sizing-invariant, so trade-knob sweeps only move equity.

gs = grid_search(
    STRATEGY,
    strategy_grid={"ema_fast": [19, 21, 56], "ema_slow": [22, 30, 76]},
    trade_grid={"leverage": [1.0, 2.0]},
    exit_grid=[None, "fixed_2pct_rr3", "chandelier_2atr"],
    data_grid={"interval": ["15", "60"]},   # reloads per spec
    base_config=STRATEGY_CONFIG, base_trading=TRADING_CONFIG, base_data=DATA_CONFIG,
)
gs.sort_values("total_return_pct", ascending=False).head(12)

In [ ]:
# HEATMAP_METRIC is a configurable knob:
# flip between Sharpe / P&L / profit_factor / any grid_search column without editing the plot.
# Best value per ema_fast × ema_slow cell, across any other swept dimension.
# Renders only when the strategy grid is being swept.

HEATMAP_METRIC = "total_pnl_bps"   # any grid_search column
if {"ema_fast", "ema_slow"}.issubset(gs.columns):
    # Diverging colour split at the metric's breakeven: P&L/Sharpe at 0, profit_factor at 1, win_rate at 0.5.
    midpoint = {"profit_factor": 1.0, "win_rate": 0.5}.get(HEATMAP_METRIC, 0.0)
    px.imshow(
        gs.pivot_table(index="ema_fast", columns="ema_slow", values=HEATMAP_METRIC, aggfunc="max"),
        color_continuous_scale="RdYlGn", color_continuous_midpoint=midpoint, aspect="auto",
        labels=dict(x="ema_slow", y="ema_fast", color=HEATMAP_METRIC),
        title=f"In-sample {HEATMAP_METRIC} — {strategy.name} | {SYMBOL} {INTERVAL}m",
    ).show()
else:
    print("Heatmap needs ema_fast × ema_slow swept in the grid above — nothing to plot.")

<a id="adp-walk-forward"></a>
### Walk-forward analysis

In [ ]:
# Walk-forward re-sweeps the strategy grid every train window.
GRID = {"ema_fast": [5, 9, 13, 17], "ema_slow": [20, 30, 40, 50]}
MIN_TRADES = 2
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    base_config=STRATEGY_CONFIG,   # non-swept knobs come from STRATEGY_CONFIG (automatic + manual)
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

In [ ]:
# Per fold: the parameters chosen in-sample (is) and their out-of-sample (oos) performance.
# Parameters that vary significantly across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

In [ ]:
# Best parameters per fold (top) + cross-fold stability summary (bottom).
# Stable across folds = trustworthy; jumpy = tend to overfit, unlikely to work next window.
# nunique==1 => the optimiser locked the same value every fold; wide min..max / large std => jumpy.
display(wf.param_stability())
wf.param_stability_summary()

In [ ]:
# Out-of-sample equity curve: the stitched test windows compounded.
# The equity path you'd have lived through re-tuning periodically on only past data.

eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
(px.line(eq, labels={"value": "equity", "index": ""}, color_discrete_sequence=["steelblue"],
         title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m")
 .update_layout(showlegend=False)
 .update_traces(hovertemplate="%{x}<br>%{y:,.2f}<extra></extra>",
                hoverlabel=dict(bgcolor="steelblue", font_color="white"))
 .show())

In [ ]:
# Walk-forward OOS trades (entries/exits), drawn with the EMAs each fold actually traded.
# Entries sit on real crosses: each fold's winning EMAs are recomputed and shown only over that fold's test window.
# The lines step at fold boundaries (the visible jump) is the re-optimisation.
# See wf.folds_frame() for the per-window parameters.

prepared_wf = df.copy()
prepared_wf["ema_fast"] = float("nan")
prepared_wf["ema_slow"] = float("nan")
for f in wf.folds:
    prep = STRATEGY(dataclasses.replace(STRATEGY_CONFIG, **f.best_params)).prepare(df)
    seg = (df.index >= f.test_start) & (df.index <= f.test_end)
    prepared_wf.loc[seg, ["ema_fast", "ema_slow"]] = prep.loc[seg, ["ema_fast", "ema_slow"]]

build_chart(prepared_wf, trades=wf.oos_trades,
            title=f"Walk-forward OOS trades — {strategy.name} | per-fold EMAs").show()

<a id="adp-monte-carlo"></a>
### Monte Carlo simulations

In [ ]:
# The spread shows the range of results that could occur due to luck, not just the outcome that actually happened.
# It answers: how much did trade ordering affect the result, and how severe could the drawdown realistically be?
# P(profitable) near 50% means the OOS edge is equivalent to noise (close to a random chance).

mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

In [ ]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.

px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"}, color_discrete_sequence=["steelblue"],
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()

# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"}, color_discrete_sequence=["steelblue"],
             title="OOS max-drawdown distribution").show()

<a id="adp-live-signals"></a>
### Live signals

In [ ]:
# Live mode runs the same strategy / config / costs as the backtest above.
# It generates signals (tells you when to enter / exit). It does not place orders.
# run() prints a clickable link to the chart, which auto-refreshes every poll_seconds.
# engine.run() blocks the execution of the rest of the notebook until stopped.

live_strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
engine = LiveEngine(
    strategy=live_strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=10,
    trading_config=TRADING_CONFIG,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{live_strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{live_strategy.name}.db"),
)
engine.run()    # to stop: interrupt the kernel/ use the Stop button